# Analisis Dataset PUBG Mobile
Notebook ini mengolah dataset PUBG Mobile untuk memprediksi `Win` (0 = tidak menang, 1 = menang). Karena target berupa dua kelas, model yang digunakan adalah Gradient Boosting Classifier. Evaluasi menggunakan Accuracy, Precision, Recall, F1-score, laporan klasifikasi, dan Confusion Matrix.


## 1. Import library

In [ ]:
# Library untuk mengolah data dan membuat grafik
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Library untuk membagi data, membuat model, dan mengevaluasi klasifikasi
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)


## 2. Data loading

In [ ]:
# Baca dataset utama
df = pd.read_csv("PUBG_Mobile.csv")

print("Ukuran dataset:", df.shape)
df.head()

## 3. Data cleaning

In [ ]:
# Cek informasi dan nilai kosong sebelum dibersihkan
df.info()

In [ ]:
print("Nilai kosong per kolom:")
print(df.isnull().sum())
print("Jumlah data duplikat:", df.duplicated().sum())

In [ ]:
# Hapus baris duplikat dan baris yang targetnya kosong
df = df.drop_duplicates()
df = df.dropna(subset=["Win"])

# Isi nilai kosong numerik dengan median dan kategorikal dengan modus
kolom_angka = df.select_dtypes(include="number").columns
kolom_kategori = df.select_dtypes(include=["object", "str", "category"]).columns

for kolom in kolom_angka:
    df[kolom] = df[kolom].fillna(df[kolom].median())

for kolom in kolom_kategori:
    if df[kolom].isnull().any():
        df[kolom] = df[kolom].fillna(df[kolom].mode()[0])

print("Ukuran setelah cleaning:", df.shape)
print("Sisa nilai kosong:", df.isnull().sum().sum())


## 4. EDA (Exploratory Data Analysis)

In [ ]:
# Ringkasan statistik data
df.describe(include="all")

In [ ]:
# Lihat jumlah data menang dan tidak menang
print(df["Win"].value_counts())
sns.countplot(data=df, x="Win")
plt.title("Jumlah Win dan Tidak Win")
plt.xlabel("Win (0 = tidak, 1 = menang)")
plt.ylabel("Jumlah data")
plt.show()

In [ ]:
# Sebaran jumlah kill
sns.histplot(data=df, x="Kills", bins=15, kde=True)
plt.title("Distribusi Kills")
plt.show()

## 5. EXDA (eksplorasi data lanjutan)

In [ ]:
# Bandingkan kill dengan damage berdasarkan status kemenangan
sns.scatterplot(data=df, x="Kills", y="Damage_Dealt", hue="Win")
plt.title("Kills dan Damage berdasarkan Win")
plt.show()

In [ ]:
# Korelasi antar kolom numerik
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Korelasi Fitur Numerik")
plt.show()

## 6. Membagi data train dan test

In [ ]:
# Win menjadi target yang ingin diprediksi. Player_ID hanya identitas pemain,
# sedangkan Match_Rank berpotensi mengandung informasi hasil akhir pertandingan.
X = df.drop(columns=["Win", "Player_ID", "Match_Rank"], errors="ignore")
y = df["Win"]

# Membagi data menjadi 80% training dan 20% testing.
# stratify menjaga perbandingan kelas Win tetap serupa di kedua bagian.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Data train:", X_train.shape)
print("Data test:", X_test.shape)
print("Distribusi Win pada data train:")
print(y_train.value_counts())
print("Distribusi Win pada data test:")
print(y_test.value_counts())

## 7. Modeling

In [ ]:
# Pisahkan kolom kategori dan angka untuk preprocessing.
kolom_kategori = X_train.select_dtypes(include=["object", "str", "category"]).columns
kolom_angka = X_train.select_dtypes(include="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("kategori", OneHotEncoder(handle_unknown="ignore"), kolom_kategori),
        ("angka", "passthrough", kolom_angka)
    ]
)

# Win adalah target klasifikasi (0 atau 1), jadi gunakan Classifier.
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

# Beri bobot lebih pada kelas yang jumlahnya sedikit agar model memperhatikan kelas menang.
bobot_train = compute_sample_weight(class_weight="balanced", y=y_train)
model.fit(X_train, y_train, model__sample_weight=bobot_train)
print("Model selesai dilatih")


Metrik evaluasi yang digunakan adalah Accuracy, Precision, Recall, F1-score, laporan klasifikasi, dan Confusion Matrix. MAE, MSE, dan R² tidak digunakan karena tujuan analisis ini adalah klasifikasi menang atau tidak menang.


In [ ]:
# Membuat prediksi untuk data testing yang belum dipakai saat pelatihan.
y_pred = model.predict(X_test)

# Menghitung metrik evaluasi klasifikasi.
akurasi = accuracy_score(y_test, y_pred)
presisi = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy :", round(akurasi, 4))
print("Precision:", round(presisi, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

print("\nLaporan klasifikasi:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Confusion matrix menunjukkan jumlah prediksi yang benar dan salah.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Tidak Menang", "Menang"],
    yticklabels=["Tidak Menang", "Menang"]
)
plt.xlabel("Prediksi Model")
plt.ylabel("Nilai Sebenarnya")
plt.title("Confusion Matrix")
plt.show()

## 9. Kesimpulan evaluasi

Model yang digunakan adalah **Gradient Boosting Classifier** karena target `Win` berisi dua kelas, yaitu 0 (tidak menang) dan 1 (menang). Hasil evaluasi dilihat dari Accuracy, Precision, Recall, F1-score, dan Confusion Matrix.

Dataset ini memiliki jumlah kelas yang tidak seimbang: data tidak menang jauh lebih banyak daripada data menang. Karena itu, Accuracy saja belum cukup untuk menilai model. Perhatikan juga Precision, Recall, F1-score, dan jumlah prediksi benar atau salah pada Confusion Matrix, terutama untuk kelas menang. Pembobotan kelas digunakan saat pelatihan agar model lebih memperhatikan kelas yang jumlahnya sedikit.

Kesimpulan akhir perlu disesuaikan dengan angka evaluasi yang muncul setelah notebook dijalankan. Model belum tentu bagus hanya karena satu metrik memiliki nilai tinggi.